In [ ]:
import numpy as np
import cupy as cp
from cupyx.scipy import ndimage as cp_ndimage
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import os
import hashlib
from datetime import datetime
import time
import tempfile

# ==========================================
# CUSTOMIZATION PARAMETERS (MODIFY THESE)
# ==========================================

# Domain parameters
xc = 0
yc = 0
W = 1
H = 1
nx = 300
ny = 300
d = 0.02  # initial pulse width

# Wave parameters
c = 1.0  # wave speed
steps = 1000#2000
cfl_factor = 0.5  # CFL stability condition factor

# Damping parameters
absorb_width = 0.25
absorb_strength = 1/nx

# Source parameters
use_ext_source = False
use_const_source = False

# Particle parameters
mass = 0.01
charge = 0.01
conjugate_charges = False
num_particles = 20


# Animation parameters
frame_step = 1  # Use every 5th frame to reduce memory
fps = 30
figsize = (10, 10)
color_factor = 1.125
animation_interval = 10

# Output parameters
output_subfolder = 'Wave Simulations'
bitrate = 3000

# ==========================================
# INITIAL CONDITION
# ==========================================

def initial_field(X, Y, x_center, y_center, pulse_width):
    #k = 2*cp.pi
  #  return cp.cos(k*Y)*(cp.exp(k*X)+cp.exp(-k*X))

#    return cp.exp(-((X-x_center)/pulse_width)**2-((Y-y_center)/pulse_width)**2)
    return cp.zeros_like(X, dtype=cp.float32)

def force_field(X, Y, time):
    #return cp.sin(2*cp.pi * time * 20)*ramp_source(X, Y, 0, 1, W/30, [-H/20, H/20])
    return cp.zeros_like(X, dtype=cp.float32)


def double_ramp(value, min_val, max_val, ramp_length, ramp_points):
    value = cp.asarray(value)

    result = cp.full_like(value, min_val, dtype=cp.float32)

    # Flat top
    mask = (value > ramp_points[0]) & (value < ramp_points[1])
    result[mask] = max_val

    # Right ramp
    mask = (value >= ramp_points[1]) & (value <= ramp_points[1] + ramp_length)
    result[mask] = ((max_val - min_val) / ramp_length *
                    (ramp_points[1] + ramp_length - value[mask]) + min_val)

    # Left ramp
    mask = (value <= ramp_points[0]) & (value >= ramp_points[0] - ramp_length)
    result[mask] = ((max_val - min_val) / ramp_length *
                    (value[mask] - ramp_points[0] + ramp_length) + min_val)

    return result


def ramp_source(X, Y, min_val, max_val, ramp_length, ramp_points):
    result = cp.zeros_like(X, dtype=cp.float32)

    mask = X <= X[0,1]

    result[mask] = double_ramp(
        Y[mask],
        min_val,
        max_val,
        ramp_length,
        ramp_points
    )

    return result
    
# ==========================================
# LAPLACIAN OPERATOR
# ==========================================

def get_laplace(Z, dx):
    """9-point stencil Laplacian"""
    Zp = cp.pad(Z, 1, mode='edge')
    
    return (
        4 * (
            Zp[2:,1:-1] +
            Zp[:-2,1:-1] +
            Zp[1:-1,2:] +
            Zp[1:-1,:-2]
        )
        +
        (
            Zp[2:,2:] +
            Zp[2:,:-2] +
            Zp[:-2,2:] +
            Zp[:-2,:-2]
        )
        -
        20 * Z
    ) / (6 * dx**2)

# ==========================================
# MATH HELPERS
# ==========================================

def bilinear_interpolate_uniform(x0, y0, dx, dy, F, px, py):
    """Bilinear interpolation"""
    ix = int((px - x0) / dx)
    iy = int((py - y0) / dy)

    ix = max(0, min(ix, F.shape[1]-2))
    iy = max(0, min(iy, F.shape[0]-2))

    tx = (px - (x0 + ix*dx)) / dx
    ty = (py - (y0 + iy*dy)) / dy

    Q11 = F[iy, ix]
    Q21 = F[iy, ix+1]
    Q12 = F[iy+1, ix]
    Q22 = F[iy+1, ix+1]

    return ((1-tx)*(1-ty)*Q11 + tx*(1-ty)*Q21 + 
            (1-tx)*ty*Q12 + tx*ty*Q22)

# ==========================================
# PARTICLE DISTRIBUTION
# ==========================================

def set_particle_distribution(simulation):
    """Set particle positions"""
    distribution = simulation['distribution']
    if distribution['type'] == "none":
        simulation["x_pos_init"] = [simulation['sim_center'][0]]
        simulation["y_pos_init"] = [simulation['sim_center'][1]]
        return simulation
        
    xc, yc = distribution['center']
    num_particles = simulation["num_particles"]
    if num_particles <= 1:
        simulation["x_pos_init"] = [xc]
        simulation["y_pos_init"] = [yc]
        return simulation

    x_particles = []
    y_particles = []
    
    if distribution['type'] == "line":
        length = distribution['length']
        angle = distribution['angle'] * cp.pi/180
        xr = cp.cos(angle) * length
        yr = cp.sin(angle) * length
        for i in range(num_particles):
            new_x = xc + xr * (i - (num_particles - 1)/2)/(num_particles-1)
            new_y = yc + yr * (i - (num_particles - 1)/2)/(num_particles-1)
            x_particles.append(float(new_x))
            y_particles.append(float(new_y))
        simulation["x_pos_init"] = cp.array(x_particles)
        simulation["y_pos_init"] = cp.array(y_particles)
        return simulation
        
    if distribution['type'] == "ring":
        radius = distribution['length']/2
        angle = distribution['angle'] * cp.pi/180
        proportion = distribution['proportion']
        rad_x = radius * proportion
        rad_y = radius / proportion
        for i in range(num_particles):
            new_x = xc + rad_x * cp.cos(2*cp.pi*i/num_particles + angle)
            new_y = yc + rad_y * cp.sin(2*cp.pi*i/num_particles + angle)
            x_particles.append(float(new_x))
            y_particles.append(float(new_y))
        simulation["x_pos_init"] = cp.array(x_particles)
        simulation["y_pos_init"] = cp.array(y_particles)
        return simulation

# ==========================================
# PARTICLE FIELD
# ==========================================

def particle_field_vectorized(index, simulation):
    """Particle field calculation"""
    X = simulation['X']
    Y = simulation['Y']
    x_pos = simulation['x_pos_new'][index]
    y_pos = simulation['y_pos_new'][index]
    z_pos = simulation['z_pos_new'][index]
    pulse_width = simulation['pulse_width_new'][index]
    
    radius = (X - x_pos)**2 + (Y - y_pos)**2 + z_pos**2
    return cp.exp(-radius/(pulse_width)**2)

def compute_all_particle_fields(simulation):
    """Compute total particle field"""
    X = simulation['X']
    Y = simulation['Y']
    num_particles = simulation['num_particles']
    
    total_field = cp.zeros_like(X, dtype=cp.float32)
    
    # Process particles in batches to avoid memory issues
    batch_size = 100
    for start_idx in range(0, num_particles, batch_size):
        end_idx = min(start_idx + batch_size, num_particles)
        batch_field = cp.zeros_like(X, dtype=cp.float32)
        
        for i in range(start_idx, end_idx):
            batch_field += simulation["particle_charges"][i] * particle_field_vectorized(i, simulation)
        
        total_field += batch_field
        # Clear batch field to free memory
        del batch_field
    
    return total_field

# ==========================================
# PARTICLE MOTION
# ==========================================

def linear_motion(index, simulation):
    """Linear motion"""
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0.0
    y_inc = 0.0
    if time > params["lin_delay"] and time < params["lin_end"]:
        x_inc = params["velocity"][0] * simulation["dt"]
        y_inc = params["velocity"][1] * simulation["dt"]
    return x_inc, y_inc

def quadratic_motion(index, simulation):
    """Quadratic motion"""
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0.0
    y_inc = 0.0
    if time > params["quad_delay"] and time < params["quad_end"]:
        accel_x = params["acceleration"][0]
        accel_y = params["acceleration"][1]
        x_inc = accel_x / 2 * simulation["dt"] * (time - params["quad_delay"])
        y_inc = accel_y / 2 * simulation["dt"] * (time - params["quad_delay"])
    return x_inc, y_inc

def oscillatory_motion(index, simulation):
    """Oscillatory motion"""
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0.0
    y_inc = 0.0
    z_inc = 0.0
    if time > params["osc_delay"] and time < params["osc_end"]:
        if not "angular_frequency" in simulation["motion_params"][index]:
            freq_x = params["frequency"][0]
            freq_y = params["frequency"][1]
            freq_z = params["frequency"][2]
            simulation["motion_params"][index]["angular_frequency"] = [2*cp.pi*freq_x, 2*cp.pi*freq_y, 2*cp.pi*freq_z]
            params = simulation["motion_params"][index]
        ang_freq_x = params["angular_frequency"][0]
        ang_freq_y = params["angular_frequency"][1]
        ang_freq_z = params["angular_frequency"][2]

        if not "oscillation_velocity" in simulation["motion_params"][index]:
            rad_x = params["amplitude"][0]
            rad_y = params["amplitude"][1]
            rad_z = params["amplitude"][2]
            simulation["motion_params"][index]["oscillation_velocity"] = [ang_freq_x*rad_x, ang_freq_y*rad_y, ang_freq_z*rad_z]
            params = simulation["motion_params"][index]
        osc_vel_x = params["oscillation_velocity"][0]
        osc_vel_y = params["oscillation_velocity"][1]
        osc_vel_z = params["oscillation_velocity"][2]
        
        phase_x = params["phase"][0]
        phase_y = params["phase"][1]
        phase_z = params["phase"][2]
        
        x_inc = osc_vel_x * cp.cos(ang_freq_x * time * (1 + 2*simulation["frame"]/steps) - phase_x) * simulation["dt"]
        y_inc = osc_vel_y * cp.cos(ang_freq_y * time * (1 + 2*simulation["frame"]/steps) - phase_y) * simulation["dt"]
        z_inc = osc_vel_z * cp.cos(ang_freq_z * time * (1 + 2*simulation["frame"]/steps) - phase_z) * simulation["dt"]
    
    return float(x_inc), float(y_inc), float(z_inc)

def particle_motion_by_forcefield(index, simulation):
    """Field-based motion"""
    current_x = simulation['x_pos'][index]
    current_y = simulation['y_pos'][index]
    field = simulation["current_field"]
    if simulation["prevent_self_interaction"]:
        field = field - particle_field_vectorized(index, simulation)
    x = simulation['X'][0,:].get()
    y = simulation['Y'][:,0].get()
    field_cpu = cp.asnumpy(field)
    field_strength = bilinear_interpolate_uniform(
        float(x[0]), float(y[0]), 
        float(simulation["dx"]), float(simulation["dx"]),
        field_cpu, float(current_x), float(current_y)
    )
    dz = float(field_strength) * simulation["particle_charges"][index] / simulation["particle_masses"][index] * simulation["dt2"]
    return dz

def update_particle_motion(index, simulation):
    """Update particle motion"""
    simulation['x_pos_new'][index] = simulation['x_pos'][index]
    simulation['y_pos_new'][index] = simulation['y_pos'][index]
    simulation['z_pos_new'][index] = simulation['z_pos'][index]
    x_inc = 0.0
    y_inc = 0.0
    z_inc = 0.0

    if "stationary" in simulation["motion_params"][index]["motion_type"]:
        pass
    else:
        if "linear" in simulation["motion_params"][index]["motion_type"]:
            x_inc_temp, y_inc_temp = linear_motion(index, simulation)
            x_inc += x_inc_temp
            y_inc += y_inc_temp
        if "quadratic" in simulation["motion_params"][index]["motion_type"]:
            x_inc_temp, y_inc_temp = quadratic_motion(index, simulation)
            x_inc += x_inc_temp
            y_inc += y_inc_temp
        if "oscillatory" in simulation["motion_params"][index]["motion_type"]:
            x_inc_temp, y_inc_temp, z_inc_temp = oscillatory_motion(index, simulation)
            x_inc += x_inc_temp
            y_inc += y_inc_temp
            z_inc += z_inc_temp
        if "field" in simulation["motion_params"][index]["motion_type"]:
            z_inc += particle_motion_by_forcefield(index, simulation)
        
    if simulation["prevent_superluminal"]:
        sq_ratio = (x_inc**2 + y_inc**2 + z_inc**2) / (simulation["c"] * simulation["dt"])**2
        if sq_ratio >= 1:
            print(f"WARNING: particle {index} at time {simulation['time']} would exceed wave speed. Capping speed.")
            slowdown = 1 / cp.sqrt(sq_ratio) * simulation["speed_cap"]
            x_inc *= slowdown
            y_inc *= slowdown
            z_inc *= slowdown
        
    simulation['x_pos_new'][index] += x_inc
    simulation['y_pos_new'][index] += y_inc
    simulation['z_pos_new'][index] += z_inc
    simulation['x_pos'][index] = simulation['x_pos_new'][index]
    simulation['y_pos'][index] = simulation['y_pos_new'][index]
    simulation['z_pos'][index] = simulation['z_pos_new'][index]
    return simulation

def set_default_motion_params(num_particles, default_type, dt):
    """Set default motion parameters"""
    motion_params = []
    for i in range(num_particles):
        if default_type == "z-wave":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [0, 0, 0],#2*cp.pi*i/num_particles],
                    "frequency" : [0, 0, 1],
                    "amplitude" : [0, 0, 0.01],
                }
        elif default_type == "x-wave":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [2*cp.pi*i/num_particles, 0, 0],
                    "frequency" : [1, 0, 0],
                    "amplitude" : [0.01, 0, 0],
                }
        elif default_type == "x-wiggle":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [1, 0, 0],
                    "amplitude" : [0.1, 0, 0],
                }
        elif default_type == "circle":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [2*cp.pi*i/num_particles  + cp.pi*i, 2*cp.pi*i/num_particles - cp.pi / 2 + cp.pi*i, 0],
                    "frequency" : [1, 1, 0],
                    "amplitude" : [0.1, 0.1, 0],
                }
        elif default_type == "linear":
            params = {
                    "motion_type" : ["linear"],
                    "velocity" : [0.1, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : i * 15 * dt,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "accelerate":
            params = {
                    "motion_type" : ["quadratic"],
                    "velocity" : [0, 0],
                    "acceleration" : [0.1, 0],
                    "lin_delay" : 0,
                    "quad_delay" : i * 1 * dt,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "interact":
            params = {
                    "motion_type" : ["field"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : cp.inf,
                    "quad_end" : cp.inf,
                    "osc_end" : cp.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "orbit":
            params = {
                "motion_type" : ["linear", "interact"],
                "velocity" : [0.01 * (-1)**i, 0],
                "acceleration" : [0, 0],
                "lin_delay" : 0,
                "quad_delay" : 0,
                "osc_delay" : 0,
                "lin_end" : cp.inf,
                "quad_end" : cp.inf,
                "osc_end" : cp.inf,
                "phase" : [0, 0],
                "frequency" : [0, 0, 0],
                "radius" : [0, 0, 0],
            }
        else:
            params = {
                "motion_type" : ["stationary"],
                "velocity" : [0, 0],
                "acceleration" : [0, 0],
                "lin_delay" : 0,
                "quad_delay" : 0,
                "osc_delay" : 0,
                "lin_end" : cp.inf,
                "quad_end" : cp.inf,
                "osc_end" : cp.inf,
                "phase" : [0, 0],
                "frequency" : [0, 0, 0],
                "radius" : [0, 0, 0],
            }
        motion_params.append(params)
    return motion_params

# ==========================================
# OPTIMIZED SIMULATION WITH DISK STORAGE
# ==========================================

def sigma_profile(sigma_max = 80, sigma_type = "circular", args = {}):
    if sigma_type == "circular":
        R = cp.sqrt((X - xc)**2 + (Y - yc)**2)
        Rmax = min(W, H) / 2
        R0 = Rmax - absorb_width
        s = cp.clip((R - R0) / absorb_width, 0, 1)
        damping = cp.exp(-absorb_strength * s**2)
        
        sigma = cp.zeros_like(R)
        sigma[s > 0] = sigma_max * s[s > 0]**4
    
    if sigma_type == "open":
        if "theta0" in args:
            theta0 = args["theta0"]
        else:
            theta0 = 0

        if "opening" in args:
            opening_angle = args["opening"]
        else:
            opening_angle = cp.pi / 4  
        
        R = cp.sqrt((X - xc)**2 + (Y - yc)**2)
        Rmax = min(W, H) / 2
        R0 = Rmax - absorb_width
        
        s = cp.clip((R - R0) / absorb_width, 0, 1)
        
        # Opening angle centered on +x direction
        theta = cp.arctan2(Y - yc, X - xc)
        
        # Smallest angular difference
        dtheta = cp.arctan2(
            cp.sin(theta - theta0),
            cp.cos(theta - theta0)
        )
        
        mask = cp.abs(dtheta) > opening_angle
        
        sigma = cp.zeros_like(R)
        sigma[mask] = sigma_max * s[mask]**4

    if sigma_type == "horizontal":
        dy = cp.abs(Y - yc)
        Ymax = H / 2
        Y0 = Ymax - 1.125 * absorb_width
        
        s = cp.clip((dy - Y0) / absorb_width, 0, 1)
        
        sigma = sigma_max * s**4

    if sigma_type == "vertical":
        dx = cp.abs(X - xc)
        
        Xmax = W / 2
        X0 = Xmax - 1.125 * absorb_width
        
        s = cp.clip((dx - X0) / absorb_width, 0, 1)
        
        sigma = sigma_max * s**4

    if sigma_type == "rect":
        sx = cp.clip((cp.abs(X - xc) - (W/2 - absorb_width)) / absorb_width, 0, 1)
        sy = cp.clip((cp.abs(Y - yc) - (H/2 - absorb_width)) / absorb_width, 0, 1)
        
        sigma = sigma_max * cp.maximum(sx, sy)**4
    return sigma

def run_simulation():
    """Run simulation with disk-based frame storage"""
    
    # ==========================================
    # GRID
    # ==========================================
    x = cp.linspace(xc - W/2, xc + W/2, nx, dtype=cp.float32)
    y = cp.linspace(yc - H/2, yc + H/2, ny, dtype=cp.float32)
    X, Y = cp.meshgrid(x, y)
    
    dx = float(x[1] - x[0])
    dt = cfl_factor * dx / c
    
    # ==========================================
    # ABSORBING BOUNDARY
    # ==========================================

    sigma = sigma_profile(sigma_max = 200, sigma_type = "circular", args = {"theta0": cp.pi, "opening" : cp.pi/8})

    # ==========================================
    # PARTICLES
    # ==========================================
    
    motion_params = set_default_motion_params(num_particles, "x-wave", dt)
    
    distribution = {
        "type": "ring",
        "center": [xc, yc],
        "length": 0.1,
        "angle": 0,
        "proportion": 1,
    }
    
    simulation = {
        "frame": 0,
        "X": X,
        "Y": Y,
        "time": 0.0,
        "dt": dt,
        "dt2": dt**2,
        "dx": dx,
        "c": c,
        "source_balance": True,
        "sim_center": [xc, yc],
        "num_particles": num_particles,
        "prevent_superluminal": True,
        "speed_cap": 0.99,
        "prevent_self_interaction": True,
        "current_field": cp.zeros_like(X, dtype=cp.float32),
        "x_pos_init": cp.zeros(num_particles, dtype=cp.float32),
        "y_pos_init": cp.zeros(num_particles, dtype=cp.float32),
        "z_pos_init": cp.zeros(num_particles, dtype=cp.float32),
        "distribution": distribution,
        "pulse_width_init": cp.ones(num_particles, dtype=cp.float32) * d,
        "particle_masses": cp.ones(num_particles, dtype=cp.float32) * mass,
        "particle_charges": cp.ones(num_particles, dtype=cp.float32) * charge,
        "x_pos": cp.zeros(num_particles, dtype=cp.float32),
        "y_pos": cp.zeros(num_particles, dtype=cp.float32),
        "z_pos": cp.zeros(num_particles, dtype=cp.float32),
        "x_pos_new": cp.zeros(num_particles, dtype=cp.float32),
        "y_pos_new": cp.zeros(num_particles, dtype=cp.float32),
        "z_pos_new": cp.zeros(num_particles, dtype=cp.float32),
        "pulse_width_new": cp.ones(num_particles, dtype=cp.float32) * d,
        "motion_params": motion_params
    }
    
    if conjugate_charges:
        for i in range(num_particles):
            simulation["particle_charges"][i] *= (-1)**i
    
    # Set particle positions
    simulation = set_particle_distribution(simulation)
    
    for i in range(num_particles):
        simulation["x_pos"][i] = simulation["x_pos_init"][i]
        simulation["y_pos"][i] = simulation["y_pos_init"][i]
        simulation["z_pos"][i] = simulation["z_pos_init"][i]
        simulation["x_pos_new"][i] = simulation["x_pos"][i]
        simulation["y_pos_new"][i] = simulation["y_pos"][i]
        simulation["z_pos_new"][i] = simulation["z_pos"][i]
    
    # ==========================================
    # INITIAL PARTICLE FIELD
    # ==========================================

    # ==========================================
    # INITIAL WAVE FIELD
    # ==========================================
    initial_particle_field = compute_all_particle_fields(simulation)
    
    Z_radiative = initial_field(X, Y, xc, yc, d)
    Z_radiative_prev = Z_radiative.copy()
    
    particle_previous = initial_particle_field.copy()
    particle_before_previous = initial_particle_field.copy()
    # ==========================================
    # PRECOMPUTE CONSTANTS
    # ==========================================
    wave_speed_sq = (c * dt)**2
    
    # ==========================================
    # CREATE MEMORY-MAPPED FILE FOR FRAMES
    # ==========================================
    # Only store every 'frame_step' frames to save space
    num_stored_frames = steps // frame_step + (1 if steps % frame_step else 0)
    frames_file = tempfile.NamedTemporaryFile(delete=False, suffix='.dat')
    frames_file.close()
    
    # Calculate required size and check if we have enough disk space
    frame_size_gb = (ny * nx * 4) / (1024**3)  # 4 bytes per float32
    total_size_gb = frame_size_gb * num_stored_frames
    
    print(f"Frame size: {frame_size_gb:.3f} GB per frame")
    print(f"Storing {num_stored_frames} frames (every {frame_step}th frame)")
    print(f"Total storage needed: {total_size_gb:.2f} GB")
    
    # Use memory-mapped file for efficient disk storage
    frames_mmap = np.memmap(
        frames_file.name,
        dtype=np.float32,
        mode='w+',
        shape=(num_stored_frames, ny, nx)
    )
    
    # ==========================================
    # MAIN LOOP
    # ==========================================
    print(f"Running simulation with {steps} steps...")
    print(f"Grid: {nx} x {ny} = {nx*ny:,} points")
    print(f"Particles: {num_particles}")
    
    frame_idx = 0
    
    for step in range(steps):
        simulation["frame"] += 1
        simulation["time"] = step * dt
        
        # Update particle positions
        for i in range(num_particles):
            simulation["current_field"] = Z_radiative
            simulation = update_particle_motion(i, simulation)
        
        # Build current particle field
        # ==========================================

        particle_total = compute_all_particle_fields(simulation)
        
        # ==========================================
        # SECOND TIME DERIVATIVE OF SOURCE
        # ==========================================
        delta2_particle = (
            particle_total
            - 2.0 * particle_previous
            + particle_before_previous
        )

        # FORCE FIELD
        current_force_field = force_field(simulation["X"], simulation["Y"], simulation["time"])
        
        # ==========================================
        # VELOCITY OF RADIATIVE FIELD
        # ==========================================
        velocity = Z_radiative - Z_radiative_prev
        
        # ==========================================
        # EVOLVE ONLY RADIATIVE FIELD
        # ==========================================
        Z_radiative_next = (
            2 * Z_radiative
            - Z_radiative_prev
            + wave_speed_sq * get_laplace(Z_radiative, dx)
            + current_force_field * simulation["dt2"]
            + delta2_particle
           - sigma * velocity * dt
        )
        
        # ==========================================
        # APPLY DAMPING ONLY TO RADIATIVE FIELD
        # ==========================================
       # Z_radiative_next *= damping
        
        # ==========================================
        # CONSTRUCT TOTAL FIELD
        # ==========================================
        Z_total = Z_radiative_next + particle_total
        
        # Store frame (only every Nth step)
        if step % frame_step == 0:
            frames_mmap[frame_idx] = cp.asnumpy(Z_radiative_next + particle_total / (1 + num_particles))
            frame_idx += 1
            
            # Progress indicator
      #      elapsed = time.time() - start_time
     #       progress = (step + 1) / steps * 100
     #       eta = elapsed / (step + 1) * (steps - step - 1) if step > 0 else 0
    #        print(f"  Step {step+1}/{steps} ({progress:.1f}%) - "
    #              f"Elapsed: {elapsed:.1f}s - ETA: {eta:.1f}s")
        
        # Advance state
        Z_radiative_prev = Z_radiative
        Z_radiative = Z_radiative_next
        
        particle_before_previous = particle_previous
        particle_previous = particle_total
        
        # Clear GPU memory periodically
        if step % 50 == 0:
            cp.get_default_memory_pool().free_all_blocks()
    
    # Flush and close memory-mapped file
    frames_mmap.flush()
    del frames_mmap
    
    return frames_file.name, cp.asnumpy(x), cp.asnumpy(y), dt, num_stored_frames

# ==========================================
# ANIMATION FUNCTIONS
# ==========================================

def generate_filename(dt):
    """Generate unique filename"""
    param_str = (f"W{W}_H{H}_nx{nx}_ny{ny}_d{d}_"
                f"c{c}_steps{steps}_cfl{cfl_factor}_"
                f"abs{absorb_strength}_{absorb_width}_"
                f"dt{dt:.6f}")
    
    param_hash = hashlib.md5(param_str.encode()).hexdigest()[:8]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"wave_{param_hash}_{timestamp}.mp4"
    
    return filename

def create_animation(frames_file, x, y, dt, num_frames):
    """Create animation from memory-mapped file"""
    
    # Open memory-mapped file for reading
    frames_mmap = np.memmap(
        frames_file,
        dtype=np.float32,
        mode='r',
        shape=(num_frames, ny, nx)
    )
    
    # Calculate color limits (sample every 10th frame to save time)
    print("Calculating color limits...")
    vmin = np.inf
    vmax = -np.inf
    
    for i in range(0, num_frames, max(1, num_frames // 20)):
        frame_min = frames_mmap[i].min()
        frame_max = frames_mmap[i].max()
        vmin = min(vmin, frame_min)
        vmax = max(vmax, frame_max)
    
    if vmin == vmax:
        print(f"WARNING: All frames have the same value ({vmin}). Setting default range.")
        vmin = -0.1
        vmax = 0.1
    else:
        vmin *= color_factor
        vmax *= color_factor
    
    print(f"Color range: [{vmin:.6f}, {vmax:.6f}]")
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    display_dt = dt * frame_step
    
    im = ax.imshow(frames_mmap[0], extent=[x[0], x[-1], y[0], y[-1]],
                  origin='lower', vmin=vmin, vmax=vmax, animated=True, cmap='RdBu_r')
    
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Wave Amplitude')
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f't = 0.000s')
    
    fig.tight_layout(pad=0.2)
    
    # Update function
    def update(frame):
        current_Z = frames_mmap[frame]
        im.set_array(current_Z)
        ax.set_title(f't = {display_dt * frame:.3f}s')
        return [im]
    
    # Create animation
    print("Creating animation...")
    ani = FuncAnimation(fig, update, frames=num_frames,
                       interval=animation_interval, blit=True)
    
    # Save animation
    filename = generate_filename(dt)
    if not os.path.exists(output_subfolder):
        os.makedirs(output_subfolder)
    
    filepath = os.path.join(output_subfolder, filename)
    print(f"Saving animation to: {filepath}")
    
    writer = FFMpegWriter(fps=fps, bitrate=bitrate)
    ani.save(filepath, writer=writer)
    
    print(f"Animation saved successfully!")
    
    # Clean up
    frames_mmap._mmap.close()
    del frames_mmap
    
    # Delete temporary file
    try:
        os.unlink(frames_file)
    except:
        pass
    
    plt.show()
    
    return ani

# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("=" * 60)
    print("GPU-Accelerated Wave Simulation (Memory-Optimized)")
    print("=" * 60)
    
    # Check GPU availability
    try:
        gpu_count = cp.cuda.runtime.getDeviceCount()
        print(f"Found {gpu_count} GPU(s)")
        for i in range(gpu_count):
            props = cp.cuda.runtime.getDeviceProperties(i)
            print(f"  GPU {i}: {props['name'].decode()} ({props['totalGlobalMem'] / 1e9:.1f} GB)")
    except Exception as e:
        print(f"Warning: Could not get GPU info: {e}")    
    # Run simulation
    t_start = time.time()
    frames_file, x, y, dt, num_frames = run_simulation()
    t_sim = time.time()
    
    sim_time = t_sim - t_start
    print(f"\nSimulation timing:")
    print(f"  Total time: {sim_time:.2f} seconds")
    print(f"  Per frame: {sim_time/steps*1000:.2f} ms")
    print(f"  Per point: {sim_time/(steps*nx*ny)*1e6:.2f} µs")
    
    # Create animation
    print("\nCreating animation...")
    t_ani0 = time.time()
    ani = create_animation(frames_file, x, y, dt, num_frames)
    t_ani = time.time()
    
    print(f"\nAnimation timing:")
    print(f"  Time to build: {(t_ani - t_ani0):.2f} seconds")
    print(f"\nTotal execution time: {(t_ani - t_start):.2f} seconds")
    
    print("\n" + "=" * 60)
    print("Simulation complete!")
    print("=" * 60)

In [ ]:
##Adicionar modo de trajetória fixa das partículas usando uma equação paramétrica vec r(t)
# Variação da velocidade da onda como função da posição
# Pressets de condição de contorno (tanto de forçar valores numa borda quanto o modo de damping)
# Soma de campo constante como "background". 
#Máscara como obstáculo reflexivo ou refratário.
#Opção de exportar posição das partículas e usar plt.scatter no plot para o caso do campo ocultado

In [ ]:
ramped = cp.asnumpy(double_ramp(
        y,
        0,
        1,
        W/30,
        [-H/20, H/20]
    ))

fig = plt.figure(figsize = (6, 3))
plt.xlabel("y")
plt.ylabel("Intensity")
plt.plot(cp.asnumpy(y), ramped)
plt.show()

In [ ]:
x = cp.linspace(xc - W/2, xc + W/2, nx, dtype=cp.float32)
y = cp.linspace(yc - H/2, yc + H/2, ny, dtype=cp.float32)
X, Y = cp.meshgrid(x, y)

def force_field(X, Y, time):
    return cp.sin(2*cp.pi * time * 20)*ramp_source(X, Y, 0, 1, W/30, [-H/20, H/20])

Z = (force_field(X, Y, 0))
plt.imshow(cp.asnumpy(Z))

In [ ]:

# ==========================================
# GRID
# ==========================================
x_i = xc - W/2
x_f = xc + W/2
y_i = yc - H/2
y_f = yc + H/2
x = cp.linspace(x_i, x_f, nx, dtype=cp.float32)
y = cp.linspace(y_i, y_f, ny, dtype=cp.float32)
X, Y = cp.meshgrid(x, y)

dx = float(x[1] - x[0])
dt = cfl_factor * dx / c

# ==========================================
# ABSORBING BOUNDARY
# ==========================================

sigma = sigma_profile(sigma_type = "rect", args = {"theta0": cp.pi, "opening": cp.pi/8})

plt.title("Sigma function")
plt.imshow(cp.asnumpy(sigma), extent=cp.asnumpy([x_i, x_f, y_i, y_f]),
                  origin='lower')
plt.colorbar()
plt.show()